In [2]:
# ============================================================================
# ENHANCED TRAINING AND PREDICTION PIPELINE FOR TELECOM CHURN
# ============================================================================
import pandas as pd
import numpy as np
import re
from datetime import datetime
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, QuantileTransformer
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, f1_score, confusion_matrix
from sklearn.feature_selection import SelectFromModel
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================
def clean_feature_names(columns):
    """Clean feature names to remove special characters for LightGBM compatibility"""
    cleaned = []
    for col in columns:
        cleaned_name = re.sub(r'[^A-Za-z0-9_]', '_', str(col))
        cleaned_name = re.sub(r'_+', '_', cleaned_name)
        cleaned_name = cleaned_name.strip('_')
        cleaned.append(cleaned_name)
    return cleaned

def create_advanced_features(df, is_train=True):
    """
    Advanced feature engineering with domain-specific insights
    """
    data = df.copy()
    
    # ========================================================================
    # 1. BASIC RATIO AND INTERACTION FEATURES
    # ========================================================================
    
    # ARPU and billing features
    data['arpu_per_year'] = data['arpu'] / 12
    data['avg_bill_per_mou'] = data['l3m_avg_bill_dura'] / (data['l3m_avg_mou'] + 1)
    data['bill_usage_ratio'] = data['cm_tot_bill_dura'] / (data['l3m_avg_bill_dura'] + 1)
    data['arpu_stability'] = data['arpu'] / (data['l3m_avg_bill_dura'].std() + 1) if is_train else 0
    
    # Data usage features
    data['flux_usage_ratio'] = data['cm_flux_use'] / (data['cm_base_plan_flux'] + data['cm_chos_plan_flux'] + 1)
    data['4g_usage_ratio'] = data['flux_4g_use'] / (data['cm_flux_use'] + 1)
    data['upload_download_ratio'] = data['flux_up_4g_sum'] / (data['flux_down_4g_sum'] + 1)
    data['avg_daily_flux'] = data['cm_flux_use'] / (data['gprs_days'] + 1)
    data['flux_efficiency'] = data['cm_flux_use'] / (data['cm_tot_bill_dura'] + 1)
    
    # ========================================================================
    # 2. TIME-BASED BEHAVIORAL PATTERNS
    # ========================================================================
    
    # Weekday vs Weekend patterns
    data['wday_flux_total'] = data['wday_day_flux'] + data['wday_night_flux']
    data['nwday_flux_total'] = data['nwday_day_flux'] + data['nwday_night_flux']
    data['wday_vs_nwday'] = data['wday_flux_total'] / (data['nwday_flux_total'] + 1)
    data['day_vs_night_wday'] = data['wday_day_flux'] / (data['wday_night_flux'] + 1)
    data['day_vs_night_nwday'] = data['nwday_day_flux'] / (data['nwday_night_flux'] + 1)
    
    # Overall day/night preference
    data['total_day_flux'] = data['wday_day_flux'] + data['nwday_day_flux']
    data['total_night_flux'] = data['wday_night_flux'] + data['nwday_night_flux']
    data['day_night_preference'] = data['total_day_flux'] / (data['total_night_flux'] + 1)
    data['is_night_user'] = (data['total_night_flux'] > data['total_day_flux']).astype(int)
    
    # ========================================================================
    # 3. ADVANCED USAGE PATTERN FEATURES
    # ========================================================================
    
    # Over-plan usage insights
    data['has_over_plan'] = (data['cm_over_plan_flux'] > 0).astype(int)
    data['over_plan_ratio'] = data['cm_over_plan_flux'] / (data['cm_base_plan_flux'] + data['cm_chos_plan_flux'] + 1)
    data['over_plan_severity'] = np.where(
        data['over_plan_ratio'] > 0.5, 2,
        np.where(data['over_plan_ratio'] > 0.2, 1, 0)
    )
    
    # Voice usage patterns
    data['local_voice_ratio'] = data['cm_local_voice_dura'] / (data['cm_tot_bill_dura'] + 1)
    data['voice_data_ratio'] = data['cm_tot_bill_dura'] / (data['cm_flux_use'] + 1)
    data['is_voice_heavy'] = (data['cm_tot_bill_dura'] > data['cm_flux_use']).astype(int)
    
    # ========================================================================
    # 4. BROADBAND AND CONNECTIVITY FEATURES
    # ========================================================================
    
    if 'bd_flux_m' in data.columns:
        data['has_broadband'] = (data['bd_flux_m'] > 0).astype(int)
        data['bd_usage_intensity'] = data['bd_flux_m'] / (data['bd_dur_m'] + 1)
        data['bd_avg_session_dur'] = data['bd_dur_m'] / (data['bd_cnt_m'] + 1)
        data['mobile_bd_ratio'] = data['cm_flux_use'] / (data['bd_flux_m'] + 1)
        data['bd_session_frequency'] = data['bd_cnt_m'] / 30  # Average sessions per day
        data['bd_engagement_score'] = data['bd_flux_m'] * data['bd_dur_m'] / 1e6
    
    # ========================================================================
    # 5. TV AND ENTERTAINMENT ENGAGEMENT
    # ========================================================================
    
    if 'user_duration_m' in data.columns:
        data['tv_engagement'] = data['user_duration_m'] / (data['login_times_m'] + 1)
        data['click_watch_ratio'] = data['click_times_m'] / (data['watch_times_m'] + 1)
        data['active_days_ratio'] = data['open_day_m'] / 30
        data['avg_session_duration'] = data['user_duration_m'] / (data['login_times_m'] + 1)
        data['engagement_intensity'] = data['watch_times_m'] / (data['open_day_m'] + 1)
        data['tv_loyalty'] = data['open_day_m'] * data['login_times_m'] / 900  # Normalized loyalty score
    
    # ========================================================================
    # 6. COMPREHENSIVE VIDEO USAGE FEATURES
    # ========================================================================
    
    video_cols = ['gm_use_dur', 'shrt_vid_use_dur', 'long_vid_use_dur', 
                  'anchor_use_dur', 'wtch_liv_use_dur', 'netdisk_use_dur']
    
    if all(col in data.columns for col in video_cols):
        # Total usage
        data['total_video_dur'] = data[video_cols].sum(axis=1)
        
        # Content type preferences
        data['game_video_ratio'] = data['gm_use_dur'] / (data['total_video_dur'] + 1)
        data['short_long_ratio'] = data['shrt_vid_use_dur'] / (data['long_vid_use_dur'] + 1)
        data['live_ondemand_ratio'] = data['wtch_liv_use_dur'] / (data['long_vid_use_dur'] + data['shrt_vid_use_dur'] + 1)
        
        # Video diversity
        data['video_diversity'] = (data[video_cols] > 0).sum(axis=1)
        
        # Day vs night patterns
        data['game_day_ratio'] = data['gm_dayt_use_dur'] / (data['gm_use_dur'] + 1)
        data['video_night_usage'] = (data['shrt_vid_ngt_use_dur'] + 
                                      data['long_vid_ngt_use_dur'] + 
                                      data['anchor_ngt_use_dur'])
        data['video_day_usage'] = (data['shrt_vid_dayt_use_dur'] + 
                                    data['long_vid_dayt_use_dur'] + 
                                    data['anchor_dayt_use_dur'])
        data['video_day_night_ratio'] = data['video_day_usage'] / (data['video_night_usage'] + 1)
        
        # Content consumption patterns
        data['is_binge_watcher'] = ((data['long_vid_use_dur'] > data['total_video_dur'].quantile(0.75)) & 
                                     (data['long_vid_use_dur'] > 0)).astype(int) if is_train else 0
        data['is_gamer'] = ((data['gm_use_dur'] > data['total_video_dur'].quantile(0.75)) & 
                            (data['gm_use_dur'] > 0)).astype(int) if is_train else 0
        
        # Streaming intensity
        data['streaming_intensity'] = (data['wtch_liv_use_dur'] + data['anchor_use_dur']) / (data['total_video_dur'] + 1)
    
    # ========================================================================
    # 7. CONTENT CONSUMPTION AND INTEREST ANALYSIS
    # ========================================================================
    
    content_cols = ['video_cnt_m', 'read_cnt_m', 'music_cnt_m', 'caijing_cnt_m', 
                    'travel_cnt_m', 'game_cnt_m', 'edu_cnt_m']
    
    if all(col in data.columns for col in content_cols):
        # Aggregated metrics
        data['total_content_cnt'] = data[content_cols].sum(axis=1)
        data['content_diversity'] = (data[content_cols] > 0).sum(axis=1)
        
        # Specific engagement metrics
        data['avg_read_time'] = data['read_time_m'] / (data['read_cnt_m'] + 1)
        data['edu_engagement'] = data['edu_time_m'] / (data['edu_cnt_m'] + 1)
        
        # Interest profiles
        data['entertainment_ratio'] = (data['video_cnt_m'] + data['music_cnt_m'] + data['game_cnt_m']) / (data['total_content_cnt'] + 1)
        data['knowledge_ratio'] = (data['read_cnt_m'] + data['edu_cnt_m'] + data['caijing_cnt_m']) / (data['total_content_cnt'] + 1)
        data['lifestyle_ratio'] = data['travel_cnt_m'] / (data['total_content_cnt'] + 1)
        
        # User persona indicators
        data['is_entertainment_focused'] = (data['entertainment_ratio'] > 0.6).astype(int)
        data['is_knowledge_seeker'] = (data['knowledge_ratio'] > 0.4).astype(int)
        data['is_diverse_user'] = (data['content_diversity'] >= 5).astype(int)
    
    # ========================================================================
    # 8. USER LABEL FEATURES
    # ========================================================================
    
    label_cols = ['hi_flux_usr_lbl', 'sev_vid_usr_lbl', 'liv_usr_lbl', 
                  'netdisk_usr_lbl', 'vid_usr_lbl', 'read_usr_lbl', 
                  'gm_usr_lbl', 'msc_usr_lbl']
    
    if all(col in data.columns for col in label_cols):
        data['total_user_labels'] = data[label_cols].sum(axis=1)
        data['is_heavy_user'] = (data['total_user_labels'] > 3).astype(int)
        data['is_video_focused'] = ((data['vid_usr_lbl'] + data['sev_vid_usr_lbl']) > 1).astype(int)
        data['is_content_creator'] = ((data['netdisk_usr_lbl'] + data['liv_usr_lbl']) > 0).astype(int)
        data['label_diversity'] = (data[label_cols] > 0).sum(axis=1)
    
    # ========================================================================
    # 9. NETWORK AND SERVICE QUALITY FEATURES
    # ========================================================================
    
    data['is_premium_user'] = ((data['is_fam_vnet_user'] == 1) | 
                               (data['is_ent_vnet_user'] == 1)).astype(int)
    data['has_unlimited'] = data['if_nulim_prod']
    data['service_stability'] = 1 - data['is_bd_status_abnormal']
    
    # Network preference
    data['is_4g_dominant'] = (data['flux_4g_use'] > 0.8 * data['cm_flux_use']).astype(int)
    
    # ========================================================================
    # 10. LOYALTY AND TENURE ANALYSIS
    # ========================================================================
    
    data['tenure_years'] = data['innet_dura'] / 365
    data['tenure_months'] = data['innet_dura'] / 30
    data['is_long_term'] = (data['innet_dura'] > 1095).astype(int)  # > 3 years
    data['is_new_customer'] = (data['innet_dura'] < 180).astype(int)  # < 6 months
    
    # Contract value and commitment
    data['contract_value'] = data['term_cont_mon'] * data['term_cont_dfee']
    data['has_contract'] = (data['term_cont_mon'] > 0).astype(int)
    data['contract_remaining_ratio'] = data['term_cont_mon'] / (data['tenure_months'] + 1)
    data['monthly_commitment'] = data['term_cont_dfee']
    data['contract_arpu_ratio'] = data['term_cont_dfee'] / (data['arpu'] + 1)
    
    # ========================================================================
    # 11. DEMOGRAPHIC AND AGE-BASED FEATURES
    # ========================================================================
    
    data['age_group'] = pd.cut(data['age'], bins=[0, 25, 35, 45, 55, 100], 
                                labels=['young', 'young_adult', 'middle', 'mature', 'senior'])
    data['is_young'] = (data['age'] < 30).astype(int)
    data['is_senior'] = (data['age'] > 55).astype(int)
    data['is_working_age'] = ((data['age'] >= 25) & (data['age'] <= 55)).astype(int)
    
    # Age-based expected behavior
    data['age_arpu_ratio'] = data['arpu'] / (data['age'] + 1)
    data['age_usage_ratio'] = data['cm_flux_use'] / (data['age'] + 1)
    
    # ========================================================================
    # 12. OUT-OF-NETWORK BEHAVIOR
    # ========================================================================
    
    data['total_out_usage'] = data['out_gprs'] + data['out_call']
    data['out_usage_ratio'] = data['total_out_usage'] / (data['cm_flux_use'] + data['cm_tot_bill_dura'] + 1)
    data['out_gprs_ratio'] = data['out_gprs'] / (data['cm_flux_use'] + 1)
    data['out_call_ratio'] = data['out_call'] / (data['cm_tot_bill_dura'] + 1)
    data['is_roamer'] = (data['total_out_usage'] > 0).astype(int)
    data['roaming_intensity'] = np.log1p(data['total_out_usage'])
    
    # ========================================================================
    # 13. ENGAGEMENT AND ACTIVITY SCORES
    # ========================================================================
    
    # Multi-dimensional engagement score
    data['engagement_score'] = (
        data['login_times_m'] * 0.2 + 
        data['click_times_m'] * 0.2 + 
        data['watch_times_m'] * 0.3 + 
        data['open_day_m'] * 0.3
    )
    
    # Activity consistency
    data['gprs_day_consistency'] = data['gprs_days'] / 30
    data['tv_day_consistency'] = data['open_day_m'] / 30
    
    # Overall digital footprint
    data['digital_footprint'] = (
        data['cm_flux_use'] / 1000 + 
        data['cm_tot_bill_dura'] / 100 + 
        data['total_content_cnt'] / 10
    )
    
    # ========================================================================
    # 14. REVENUE AND VALUE METRICS
    # ========================================================================
    
    data['revenue_per_gb'] = data['arpu'] / ((data['cm_flux_use'] / 1024) + 1)
    data['revenue_per_minute'] = data['arpu'] / (data['l3m_avg_mou'] + 1)
    data['lifetime_value'] = data['arpu'] * data['tenure_months']
    data['value_efficiency'] = data['cm_flux_use'] / (data['arpu'] + 1)
    
    # Customer profitability indicators
    data['high_value_customer'] = ((data['arpu'] > data['arpu'].quantile(0.75)) & 
                                    (data['tenure_years'] > 2)).astype(int) if is_train else 0
    data['low_value_customer'] = ((data['arpu'] < data['arpu'].quantile(0.25)) & 
                                   (data['tenure_years'] < 1)).astype(int) if is_train else 0
    
    # ========================================================================
    # 15. POLYNOMIAL AND INTERACTION FEATURES
    # ========================================================================
    
    # Key polynomial features
    data['arpu_squared'] = data['arpu'] ** 2
    data['flux_squared'] = np.log1p(data['cm_flux_use']) ** 2
    data['tenure_squared'] = data['tenure_years'] ** 2
    
    # Critical interactions
    data['arpu_tenure_interaction'] = data['arpu'] * data['tenure_years']
    data['flux_arpu_interaction'] = data['cm_flux_use'] * data['arpu'] / 1000
    data['age_tenure_interaction'] = data['age'] * data['tenure_years']
    data['contract_tenure_interaction'] = data['contract_value'] * data['tenure_years']
    
    # ========================================================================
    # 16. STATISTICAL AGGREGATIONS
    # ========================================================================
    
    # Create grouped statistics for similar features
    flux_features = ['wday_day_flux', 'wday_night_flux', 'nwday_day_flux', 'nwday_night_flux']
    if all(col in data.columns for col in flux_features):
        data['flux_mean'] = data[flux_features].mean(axis=1)
        data['flux_std'] = data[flux_features].std(axis=1)
        data['flux_max'] = data[flux_features].max(axis=1)
        data['flux_min'] = data[flux_features].min(axis=1)
        data['flux_range'] = data['flux_max'] - data['flux_min']
        data['flux_cv'] = data['flux_std'] / (data['flux_mean'] + 1)  # Coefficient of variation
    
    # ========================================================================
    # 17. CHURN RISK INDICATORS
    # ========================================================================
    
    # Declining usage indicators
    data['usage_decline_risk'] = (
        (data['flux_usage_ratio'] < 0.5) & 
        (data['gprs_day_consistency'] < 0.5)
    ).astype(int)
    
    # Contract expiry risk
    data['contract_expiry_risk'] = (
        (data['term_cont_mon'] > 0) & 
        (data['term_cont_mon'] < 3)
    ).astype(int)
    
    # Low engagement risk
    data['low_engagement_risk'] = (
        (data['engagement_score'] < data['engagement_score'].quantile(0.25)) if is_train else 0
    ).astype(int) if is_train else 0
    
    # High cost sensitivity
    data['cost_sensitive'] = (
        (data['over_plan_ratio'] > 0.3) & 
        (data['arpu'] < data['arpu'].median() if is_train else 50)
    ).astype(int)
    
    # ========================================================================
    # 18. HANDLE CATEGORICAL FEATURES
    # ========================================================================
    
    if 'age_group' in data.columns:
        le = LabelEncoder()
        data['age_group_encoded'] = le.fit_transform(data['age_group'].astype(str))
    
    # ========================================================================
    # 19. NORMALIZATION FEATURES (Log transforms for skewed distributions)
    # ========================================================================
    
    skewed_features = ['arpu', 'cm_flux_use', 'cm_tot_bill_dura', 'innet_dura']
    for feat in skewed_features:
        if feat in data.columns:
            data[f'{feat}_log'] = np.log1p(data[feat])
            data[f'{feat}_sqrt'] = np.sqrt(data[feat])
    
    return data

def get_feature_columns(df):
    """Get the list of feature columns to use for modeling"""
    exclude_cols = ['Unnamed: 0', 'id', 'label', 'age_group']
    feature_cols = [col for col in df.columns if col not in exclude_cols]
    return feature_cols

def reduce_features_by_importance(X, y, model, threshold=0.001):
    """
    Reduce features based on feature importance
    """
    print(f"\nPerforming feature selection (threshold={threshold})...")
    selector = SelectFromModel(model, threshold=threshold, prefit=False)
    selector.fit(X, y)
    selected_features = X.columns[selector.get_support()].tolist()
    print(f"Selected {len(selected_features)} features out of {len(X.columns)}")
    return selected_features

# ============================================================================
# LOAD AND PREPARE TRAINING DATA
# ============================================================================
print("=" * 80)
print("LOADING TRAINING DATA")
print("=" * 80)

df_train = pd.read_csv('./train.csv')
print(f"Training data shape: {df_train.shape}")
print(f"\nClass distribution:")
print(df_train['label'].value_counts())
print(f"Class ratio: {df_train['label'].value_counts()[0] / df_train['label'].value_counts()[1]:.2f}:1")

# Check for missing values
print(f"\nMissing values summary:")
missing_summary = df_train.isnull().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)
if len(missing_summary) > 0:
    print(missing_summary.head(10))
else:
    print("No missing values found")

# ============================================================================
# ADVANCED FEATURE ENGINEERING
# ============================================================================
print("\n" + "=" * 80)
print("CREATING ADVANCED FEATURES")
print("=" * 80)

df_train_processed = create_advanced_features(df_train, is_train=True)

# Get feature columns
feature_cols = get_feature_columns(df_train_processed)
print(f"Total features after engineering: {len(feature_cols)}")

# Clean feature names
original_feature_cols = feature_cols.copy()
cleaned_feature_cols = clean_feature_names(feature_cols)
feature_name_mapping = dict(zip(cleaned_feature_cols, original_feature_cols))

# Prepare X and y
X = df_train_processed[feature_cols].copy()
X.columns = cleaned_feature_cols
y = df_train_processed['label']

# Handle missing values
print("\nHandling missing values...")
for col in X.columns:
    if X[col].dtype in ['float64', 'int64']:
        X[col].fillna(X[col].median(), inplace=True)
    else:
        X[col].fillna(X[col].mode()[0] if len(X[col].mode()) > 0 else 0, inplace=True)

# Replace infinities
X.replace([np.inf, -np.inf], 0, inplace=True)

print(f"Final feature matrix shape: {X.shape}")
print(f"\nFeature statistics:")
print(f"  Numeric features: {X.select_dtypes(include=[np.number]).shape[1]}")
print(f"  Memory usage: {X.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# ============================================================================
# FEATURE SCALING (Optional - for some models)
# ============================================================================
print("\n" + "=" * 80)
print("FEATURE SCALING")
print("=" * 80)

# Create scaled version for models that benefit from it
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)
print("Features scaled using StandardScaler")

# ============================================================================
# TRAIN-VALIDATION SPLIT
# ============================================================================
print("\n" + "=" * 80)
print("SPLITTING DATA FOR VALIDATION")
print("=" * 80)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.12, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Training class distribution: {y_train.value_counts().to_dict()}")
print(f"Validation class distribution: {y_val.value_counts().to_dict()}")


LOADING TRAINING DATA
Training data shape: (59904, 88)

Class distribution:
label
0    44904
1    15000
Name: count, dtype: int64
Class ratio: 2.99:1

Missing values summary:
gm_use_dur               47451
gm_dayt_use_dur          47451
gm_ngt_use_dur           47451
shrt_vid_use_dur         47451
shrt_vid_dayt_use_dur    47451
shrt_vid_ngt_use_dur     47451
long_vid_use_dur         47451
long_vid_dayt_use_dur    47451
long_vid_ngt_use_dur     47451
anchor_use_dur           47451
dtype: int64

CREATING ADVANCED FEATURES
Total features after engineering: 208

Handling missing values...
Final feature matrix shape: (59904, 208)

Feature statistics:
  Numeric features: 208
  Memory usage: 95.06 MB

FEATURE SCALING
Features scaled using StandardScaler

SPLITTING DATA FOR VALIDATION
Training set: (52715, 208)
Validation set: (7189, 208)
Training class distribution: {0: 39515, 1: 13200}
Validation class distribution: {0: 5389, 1: 1800}


In [3]:

# ============================================================================
# ADVANCED MODEL TRAINING WITH CROSS-VALIDATION
# ============================================================================
print("\n" + "=" * 80)
print("TRAINING ADVANCED MODELS WITH CROSS-VALIDATION")
print("=" * 80)

# Calculate scale_pos_weight
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Scale pos weight: {scale_pos_weight:.2f}")

# Setup cross-validation
n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

# Store results
results = {}

# ============================================================================
# MODEL 1: LIGHTGBM WITH HYPERPARAMETER TUNING
# ============================================================================
print("\n" + "-" * 80)
print("1. TRAINING LIGHTGBM")
print("-" * 80)

lgb_params = {
    'n_estimators': 1000,
    'max_depth': 10,
    'learning_rate': 0.03,
    'num_leaves': 40,
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'min_child_samples': 30,
    'reg_alpha': 0.3,
    'reg_lambda': 0.3,
    'class_weight': 'balanced',
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
    'force_col_wise': True,
    'min_split_gain': 0.01,
    'min_child_weight': 0.001
}
lgb_params_enhanced = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'n_estimators': 1500,  # Increased
    'max_depth': 12,       # Slightly deeper
    'learning_rate': 0.02, # Lower learning rate
    'num_leaves': 60,      # More leaves
    'subsample': 0.85,
    'subsample_freq': 1,
    'colsample_bytree': 0.8,
    'min_child_samples': 25,
    'min_child_weight': 0.001,
    'min_split_gain': 0.005,
    'reg_alpha': 0.4,
    'reg_lambda': 0.4,
    'class_weight': 'balanced',
    'random_state': 42,
    'feature_fraction_seed': 42,
    'bagging_seed': 42,
    'drop_seed': 42,
    'data_random_seed': 42,
    'extra_trees': True,      # Extra randomization
    'path_smooth': 0.1,       # Path smoothing
    'verbose': -1,
    'force_col_wise': True
}

lgb_model = lgb.LGBMClassifier(**lgb_params_enhanced)

# # Cross-validation
# print("Performing 5-fold cross-validation...")
# cv_scores_lgb = cross_val_score(lgb_model, X_train, y_train, cv=skf, 
#                                  scoring='roc_auc', n_jobs=-1)
# print(f"CV ROC-AUC scores: {cv_scores_lgb}")
# print(f"Mean CV ROC-AUC: {cv_scores_lgb.mean():.4f} (+/- {cv_scores_lgb.std() * 2:.4f})")

# Train on full training set with early stopping
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

y_pred_lgb = lgb_model.predict(X_val)
y_proba_lgb = lgb_model.predict_proba(X_val)[:, 1]

lgb_acc = accuracy_score(y_val, y_pred_lgb)
lgb_auc = roc_auc_score(y_val, y_proba_lgb)
lgb_f1 = f1_score(y_val, y_pred_lgb)

print(f"\nValidation Metrics:")
print(f"  Accuracy: {lgb_acc:.4f}")
print(f"  ROC-AUC: {lgb_auc:.4f}")
print(f"  F1-Score: {lgb_f1:.4f}")

results['LightGBM'] = {
    'model': lgb_model,
    # 'cv_scores': cv_scores_lgb,
    'val_auc': lgb_auc,
    'val_f1': lgb_f1,
    'predictions': y_pred_lgb,
    'probabilities': y_proba_lgb
}



TRAINING ADVANCED MODELS WITH CROSS-VALIDATION
Scale pos weight: 2.99

--------------------------------------------------------------------------------
1. TRAINING LIGHTGBM
--------------------------------------------------------------------------------

Validation Metrics:
  Accuracy: 0.7703
  ROC-AUC: 0.8456
  F1-Score: 0.6226


In [4]:

# ============================================================================
# MODEL 2: XGBOOST WITH ADVANCED PARAMETERS
# ============================================================================
print("\n" + "-" * 80)
print("2. TRAINING XGBOOST")
print("-" * 80)

xgb_params = {
    'n_estimators': 1000,
    'max_depth': 7,
    'learning_rate': 0.03,
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'colsample_bylevel': 0.7,
    'colsample_bynode': 0.7,
    'min_child_weight': 5,
    'gamma': 0.1,
    'reg_alpha': 0.3,
    'reg_lambda': 0.3,
    'scale_pos_weight': scale_pos_weight,
    'random_state': 42,
    'n_jobs': -1,
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'max_bin': 256
}
xgb_params_enhanced = {
    'n_estimators': 1500,
    'max_depth': 8,
    'learning_rate': 0.02,
    'subsample': 0.85,
    'colsample_bytree': 0.8,
    'colsample_bylevel': 0.8,
    'colsample_bynode': 0.8,
    'min_child_weight': 3,
    'gamma': 0.05,
    'reg_alpha': 0.4,
    'reg_lambda': 0.4,
    'scale_pos_weight': scale_pos_weight,
    'random_state': 42,
    'tree_method': 'hist',
    'grow_policy': 'depthwise',
    'max_bin': 512,          # Increased bins
    # 'sampling_method': 'gradient_based',
    'eval_metric': 'auc'
}

xgb_model = xgb.XGBClassifier(**xgb_params_enhanced)

# # Cross-validation
# print("Performing 5-fold cross-validation...")
# cv_scores_xgb = cross_val_score(xgb_model, X_train, y_train, cv=skf, 
#                                  scoring='roc_auc', n_jobs=-1)
# print(f"CV ROC-AUC scores: {cv_scores_xgb}")
# print(f"Mean CV ROC-AUC: {cv_scores_xgb.mean():.4f} (+/- {cv_scores_xgb.std() * 2:.4f})")

# Train on full training set with early stopping
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    # early_stopping_rounds=50,
    verbose=False
)

y_pred_xgb = xgb_model.predict(X_val)
y_proba_xgb = xgb_model.predict_proba(X_val)[:, 1]

xgb_acc = accuracy_score(y_val, y_pred_xgb)
xgb_auc = roc_auc_score(y_val, y_proba_xgb)
xgb_f1 = f1_score(y_val, y_pred_xgb)

print(f"\nValidation Metrics:")
print(f"  Accuracy: {xgb_acc:.4f}")
print(f"  ROC-AUC: {xgb_auc:.4f}")
print(f"  F1-Score: {xgb_f1:.4f}")

results['XGBoost'] = {
    'model': xgb_model,
    # 'cv_scores': cv_scores_xgb,
    'val_auc': xgb_auc,
    'val_f1': xgb_f1,
    'predictions': y_pred_xgb,
    'probabilities': y_proba_xgb
}



--------------------------------------------------------------------------------
2. TRAINING XGBOOST
--------------------------------------------------------------------------------

Validation Metrics:
  Accuracy: 0.8000
  ROC-AUC: 0.8493
  F1-Score: 0.6330


In [5]:

# ============================================================================
# MODEL 3: CATBOOST
# ============================================================================
print("\n" + "-" * 80)
print("3. TRAINING CATBOOST")
print("-" * 80)

catboost_params = {
    'iterations': 1000,
    'depth': 8,
    'learning_rate': 0.03,
    'l2_leaf_reg': 3,
    'subsample': 0.8,
    'colsample_bylevel': 0.8,
    'min_data_in_leaf': 20,
    'auto_class_weights': 'Balanced',
    'random_state': 42,
    'thread_count': -1,
    'verbose': False,
    'early_stopping_rounds': 50,
    'eval_metric': 'AUC',
    'border_count': 128
}

catboost_model = CatBoostClassifier(**catboost_params)

# # Cross-validation
# print("Performing 5-fold cross-validation...")
# cv_scores_cat = cross_val_score(catboost_model, X_train, y_train, cv=skf, 
#                                 scoring='roc_auc', n_jobs=-1)
# print(f"CV ROC-AUC scores: {cv_scores_cat}")
# print(f"Mean CV ROC-AUC: {cv_scores_cat.mean():.4f} (+/- {cv_scores_cat.std() * 2:.4f})")

# Train on full training set
catboost_model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    verbose=False
)

y_pred_cat = catboost_model.predict(X_val)
y_proba_cat = catboost_model.predict_proba(X_val)[:, 1]

cat_acc = accuracy_score(y_val, y_pred_cat)
cat_auc = roc_auc_score(y_val, y_proba_cat)
cat_f1 = f1_score(y_val, y_pred_cat)

print(f"\nValidation Metrics:")
print(f"  Accuracy: {cat_acc:.4f}")
print(f"  ROC-AUC: {cat_auc:.4f}")
print(f"  F1-Score: {cat_f1:.4f}")

results['CatBoost'] = {
    'model': catboost_model,
    # 'cv_scores': cv_scores_cat,
    'val_auc': cat_auc,
    'val_f1': cat_f1,
    'predictions': y_pred_cat,
    'probabilities': y_proba_cat
}



--------------------------------------------------------------------------------
3. TRAINING CATBOOST
--------------------------------------------------------------------------------

Validation Metrics:
  Accuracy: 0.7740
  ROC-AUC: 0.8460
  F1-Score: 0.6289


In [6]:

# ============================================================================
# MODEL 4: RANDOM FOREST WITH OPTIMIZED PARAMETERS
# ============================================================================
print("\n" + "-" * 80)
print("4. TRAINING RANDOM FOREST")
print("-" * 80)

rf_params = {
    'n_estimators': 500,
    'max_depth': 15,
    'min_samples_split': 10,
    'min_samples_leaf': 5,
    'max_features': 'sqrt',
    'bootstrap': True,
    'class_weight': 'balanced',
    'random_state': 42,
    'n_jobs': -1,
    'criterion': 'gini',
    'max_samples': 0.8
}

rf_model = RandomForestClassifier(**rf_params)

# # Cross-validation
# print("Performing 5-fold cross-validation...")
# cv_scores_rf = cross_val_score(rf_model, X_train, y_train, cv=skf, 
#                                scoring='roc_auc', n_jobs=-1)
# print(f"CV ROC-AUC scores: {cv_scores_rf}")
# print(f"Mean CV ROC-AUC: {cv_scores_rf.mean():.4f} (+/- {cv_scores_rf.std() * 2:.4f})")

# Train on full training set
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_val)
y_proba_rf = rf_model.predict_proba(X_val)[:, 1]

rf_acc = accuracy_score(y_val, y_pred_rf)
rf_auc = roc_auc_score(y_val, y_proba_rf)
rf_f1 = f1_score(y_val, y_pred_rf)

print(f"\nValidation Metrics:")
print(f"  Accuracy: {rf_acc:.4f}")
print(f"  ROC-AUC: {rf_auc:.4f}")
print(f"  F1-Score: {rf_f1:.4f}")

results['RandomForest'] = {
    'model': rf_model,
    # 'cv_scores': cv_scores_rf,
    'val_auc': rf_auc,
    'val_f1': rf_f1,
    'predictions': y_pred_rf,
    'probabilities': y_proba_rf
}



--------------------------------------------------------------------------------
4. TRAINING RANDOM FOREST
--------------------------------------------------------------------------------

Validation Metrics:
  Accuracy: 0.7790
  ROC-AUC: 0.8360
  F1-Score: 0.6142


In [7]:

# # ============================================================================
# # ENSEMBLE METHODS
# # ============================================================================
# print("\n" + "=" * 80)
# print("CREATING ENSEMBLE MODELS")
# print("=" * 80)

# # Weighted Average Ensemble
# print("\n" + "-" * 80)
# print("1. WEIGHTED AVERAGE ENSEMBLE")
# print("-" * 80)

# # Use validation AUC scores as weights
# weights = np.array([
#     results['LightGBM']['val_auc'],
#     results['XGBoost']['val_auc'],
#     results['CatBoost']['val_auc'],
#     results['RandomForest']['val_auc']
# ])
# weights = weights / weights.sum()  # Normalize weights

# print(f"Ensemble weights: LGB={weights[0]:.3f}, XGB={weights[1]:.3f}, "
#       f"CAT={weights[2]:.3f}, RF={weights[3]:.3f}")

# # Create weighted ensemble predictions
# ensemble_proba = (weights[0] * results['LightGBM']['probabilities'] +
#                  weights[1] * results['XGBoost']['probabilities'] +
#                  weights[2] * results['CatBoost']['probabilities'] +
#                  weights[3] * results['RandomForest']['probabilities'])

# ensemble_pred = (ensemble_proba > 0.5).astype(int)

# ensemble_acc = accuracy_score(y_val, ensemble_pred)
# ensemble_auc = roc_auc_score(y_val, ensemble_proba)
# ensemble_f1 = f1_score(y_val, ensemble_pred)

# print(f"\nWeighted Ensemble Metrics:")
# print(f"  Accuracy: {ensemble_acc:.4f}")
# print(f"  ROC-AUC: {ensemble_auc:.4f}")
# print(f"  F1-Score: {ensemble_f1:.4f}")

# results['WeightedEnsemble'] = {
#     'val_auc': ensemble_auc,
#     'val_f1': ensemble_f1,
#     'predictions': ensemble_pred,
#     'probabilities': ensemble_proba,
#     'weights': weights
# }


In [12]:

# Voting Classifier
print("\n" + "-" * 80)
print("2. VOTING CLASSIFIER")
print("-" * 80)

voting_estimators = [
    # ('lgb', lgb_model),
    ('xgb', xgb_model),
    ('cat', catboost_model),
    ('rf', rf_model)
]

voting_clf = VotingClassifier(
    estimators=voting_estimators,
    voting='soft',
    n_jobs=-1
)

# # Cross-validation for voting classifier
# cv_scores_voting = cross_val_score(voting_clf, X_train, y_train, cv=skf, 
#                                   scoring='roc_auc', n_jobs=-1)
# print(f"CV ROC-AUC scores: {cv_scores_voting}")
# print(f"Mean CV ROC-AUC: {cv_scores_voting.mean():.4f} (+/- {cv_scores_voting.std() * 2:.4f})")

# Train voting classifier
voting_clf.fit(X_train, y_train)

y_pred_voting = voting_clf.predict(X_val)
y_proba_voting = voting_clf.predict_proba(X_val)[:, 1]

voting_acc = accuracy_score(y_val, y_pred_voting)
voting_auc = roc_auc_score(y_val, y_proba_voting)
voting_f1 = f1_score(y_val, y_pred_voting)

print(f"\nVoting Classifier Metrics:")
print(f"  Accuracy: {voting_acc:.4f}")
print(f"  ROC-AUC: {voting_auc:.4f}")
print(f"  F1-Score: {voting_f1:.4f}")

results['VotingClassifier'] = {
    'model': voting_clf,
    # 'cv_scores': cv_scores_voting,
    'val_auc': voting_auc,
    'val_f1': voting_f1,
    'predictions': y_pred_voting,
    'probabilities': y_proba_voting
}




--------------------------------------------------------------------------------
2. VOTING CLASSIFIER
--------------------------------------------------------------------------------

Voting Classifier Metrics:
  Accuracy: 0.7891
  ROC-AUC: 0.8490
  F1-Score: 0.6306


In [ ]:

# ============================================================================
# STACKING CLASSIFIER
# ============================================================================

# Voting Classifier
print("\n" + "-" * 80)
print("3. STACKING CLASSIFIER")
print("-" * 80)

from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

stacking_clf = StackingClassifier(
    estimators=voting_estimators,
    final_estimator=RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        random_state=42,
        max_depth=3  # Keep it simple
    ),
    cv=5,
    n_jobs=-1,
    passthrough=False
)

# # Cross-validation for stacking classifier
# cv_scores_stacking = cross_val_score(stacking_clf, X_train, y_train, cv=skf, 
#                                   scoring='roc_auc', n_jobs=-1)
# print(f"CV ROC-AUC scores: {cv_scores_stacking}")
# print(f"Mean CV ROC-AUC: {cv_scores_stacking.mean():.4f} (+/- {cv_scores_stacking.std() * 2:.4f})")

# Train stacking classifier
stacking_clf.fit(X_train, y_train)

y_pred_stacking = stacking_clf.predict(X_val)
y_proba_stacking = stacking_clf.predict_proba(X_val)[:, 1]

stacking_acc = accuracy_score(y_val, y_pred_stacking)
stacking_auc = roc_auc_score(y_val, y_proba_stacking)
stacking_f1 = f1_score(y_val, y_pred_stacking)

print(f"\nstacking Classifier Metrics:")
print(f"  Accuracy: {stacking_acc:.4f}")
print(f"  ROC-AUC: {stacking_auc:.4f}")
print(f"  F1-Score: {stacking_f1:.4f}")

results['stackingClassifier'] = {
    'model': stacking_clf,
    # 'cv_scores': cv_scores_stacking,
    'val_auc': stacking_auc,
    'val_f1': stacking_f1,
    'predictions': y_pred_stacking,
    'probabilities': y_proba_stacking
}




--------------------------------------------------------------------------------
3. STACKING CLASSIFIER
--------------------------------------------------------------------------------


In [ ]:

# ============================================================================
# MODEL COMPARISON AND SELECTION
# ============================================================================
print("\n" + "=" * 80)
print("MODEL COMPARISON SUMMARY")
print("=" * 80)

# Create comparison dataframe
comparison_data = []
for model_name, model_results in results.items():
    if 'cv_scores' in model_results:
        cv_mean = model_results['cv_scores'].mean()
        cv_std = model_results['cv_scores'].std()
    else:
        cv_mean = cv_std = np.nan
    
    comparison_data.append({
        'Model': model_name,
        'CV_AUC_Mean': cv_mean,
        'CV_AUC_Std': cv_std,
        'Val_AUC': model_results['val_auc'],
        'Val_F1': model_results['val_f1']
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('Val_AUC', ascending=False)

print("\nModel Performance Comparison:")
print("=" * 60)
for _, row in comparison_df.iterrows():
    if pd.isna(row['CV_AUC_Mean']):
        print(f"{row['Model']:20s} | Val AUC: {row['Val_AUC']:.4f} | Val F1: {row['Val_F1']:.4f}")
    else:
        print(f"{row['Model']:20s} | CV AUC: {row['CV_AUC_Mean']:.4f}±{row['CV_AUC_Std']:.4f} | "
              f"Val AUC: {row['Val_AUC']:.4f} | Val F1: {row['Val_F1']:.4f}")

# Select best model
best_model_name = comparison_df.iloc[0]['Model']
best_model_results = results[best_model_name]

print(f"\n🏆 BEST MODEL: {best_model_name}")
print(f"   Validation AUC: {best_model_results['val_auc']:.4f}")
print(f"   Validation F1:  {best_model_results['val_f1']:.4f}")

# ============================================================================
# FEATURE IMPORTANCE ANALYSIS
# ============================================================================
print("\n" + "=" * 80)
print("FEATURE IMPORTANCE ANALYSIS")
print("=" * 80)

def get_feature_importance(model, feature_names, model_type):
    """Extract feature importance from different model types"""
    if hasattr(model, 'feature_importances_'):
        importance = model.feature_importances_
    elif hasattr(model, 'coef_'):
        importance = abs(model.coef_[0])
    else:
        return None
    
    feature_importance = pd.DataFrame({
        'feature': feature_names,
        'importance': importance
    }).sort_values('importance', ascending=False)
    
    return feature_importance

# Get feature importance from best tree-based models
models_for_importance = ['LightGBM', 'XGBoost', 'CatBoost', 'RandomForest']

print("Top 20 Most Important Features:")
print("-" * 50)

for model_name in models_for_importance:
    if model_name in results and 'model' in results[model_name]:
        importance_df = get_feature_importance(
            results[model_name]['model'], 
            X.columns, 
            model_name
        )
        
        if importance_df is not None:
            print(f"\n{model_name}:")
            top_features = importance_df.head(10)
            for idx, row in top_features.iterrows():
                original_name = feature_name_mapping.get(row['feature'], row['feature'])
                print(f"  {row['feature'][:30]:30s} | {row['importance']:.4f}")

# ============================================================================
# SAVE MODELS AND RESULTS
# ============================================================================
print("\n" + "=" * 80)
print("SAVING MODELS AND RESULTS")
print("=" * 80)

import pickle
import os

# Create models directory
os.makedirs('models', exist_ok=True)

# Save best model
best_model = best_model_results['model'] if 'model' in best_model_results else None
if best_model is not None:
    with open(f'models/best_model_{best_model_name.lower()}.pkl', 'wb') as f:
        pickle.dump(best_model, f)
    print(f"✅ Best model ({best_model_name}) saved to models/")

# Save feature columns and preprocessing objects
with open('models/feature_columns.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

with open('models/feature_name_mapping.pkl', 'wb') as f:
    pickle.dump(feature_name_mapping, f)

with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save results summary
results_summary = {
    'model_comparison': comparison_df.to_dict('records'),
    'best_model_name': best_model_name,
    'best_model_auc': best_model_results['val_auc'],
    'best_model_f1': best_model_results['val_f1'],
    'feature_count': len(feature_cols),
    'training_samples': len(X_train),
    'validation_samples': len(X_val)
}

with open('models/training_results.pkl', 'wb') as f:
    pickle.dump(results_summary, f)

print(f"✅ Feature preprocessing objects saved")
print(f"✅ Training results summary saved")

print("\n" + "=" * 80)
print("TRAINING COMPLETED SUCCESSFULLY!")
print("=" * 80)
print(f"🎯 Best Model: {best_model_name}")
print(f"📊 Validation AUC: {best_model_results['val_auc']:.4f}")
print(f"📈 Validation F1: {best_model_results['val_f1']:.4f}")
print(f"💾 Models saved to 'models/' directory")
print("=" * 80)


MODEL COMPARISON SUMMARY

Model Performance Comparison:
stackingClassifier   | Val AUC: 0.8496 | Val F1: 0.6253
XGBoost              | Val AUC: 0.8493 | Val F1: 0.6330
VotingClassifier     | Val AUC: 0.8493 | Val F1: 0.6336
CatBoost             | Val AUC: 0.8460 | Val F1: 0.6289
LightGBM             | Val AUC: 0.8456 | Val F1: 0.6226
RandomForest         | Val AUC: 0.8360 | Val F1: 0.6142

🏆 BEST MODEL: stackingClassifier
   Validation AUC: 0.8496
   Validation F1:  0.6253

FEATURE IMPORTANCE ANALYSIS
Top 20 Most Important Features:
--------------------------------------------------

LightGBM:
  is_bd_tv                       | 1361.0000
  is_10g_pon                     | 1307.0000
  bd_dur_m                       | 910.0000
  bd_usage_intensity             | 883.0000
  cm_flux_tot_cnt                | 826.0000
  local_voice_ratio              | 779.0000
  is_fam_vnet_user               | 764.0000
  bd_avg_session_dur             | 755.0000
  cm_base_plan_flux              | 719.0000


In [ ]:

# ============================================================================
# LOAD AND PREDICT TEST DATA
# ============================================================================

print("\n" + "=" * 80)
print("LOADING TEST DATA")
print("=" * 80)

df_test = pd.read_csv('./testB.csv')
print(f"Test data shape: {df_test.shape}")

# Store IDs
test_ids = df_test['id'].copy()

# Create features
print("\nCreating features for test data...")
df_test_processed = create_advanced_features(df_test, is_train=False)

# Prepare test features
X_test = df_test_processed[feature_cols].copy()
X_test.columns = cleaned_feature_cols

# Fill missing values
print("Handling missing values in test data...")
for col in X_test.columns:
    if col in X.columns:
        if X_test[col].dtype in ['float64', 'int64']:
            X_test[col].fillna(X[col].median(), inplace=True)
        else:
            X_test[col].fillna(X[col].mode()[0] if len(X[col].mode()) > 0 else 0, inplace=True)
    else:
        X_test[col].fillna(0, inplace=True)

# Replace infinities
X_test.replace([np.inf, -np.inf], 0, inplace=True)

print(f"Test feature matrix shape: {X_test.shape}")

# ============================================================================
# MAKE PREDICTIONS
# ============================================================================

print("\n" + "=" * 80)
print("MAKING PREDICTIONS")
print("=" * 80)

predictions = best_model.predict(X_test)
predictions_proba = best_model.predict_proba(X_test)[:, 1]

print(f"Predictions distribution:")
print(f"Class 0: {(predictions == 0).sum()} ({(predictions == 0).sum() / len(predictions) * 100:.1f}%)")
print(f"Class 1: {(predictions == 1).sum()} ({(predictions == 1).sum() / len(predictions) * 100:.1f}%)")

# ============================================================================
# SAVE RESULTS
# ============================================================================

print("\n" + "=" * 80)
print("SAVING RESULTS")
print("=" * 80)

# Create submission dataframe
submission = pd.DataFrame({
    'id': test_ids,
    'label': predictions
})

# Save to CSV
output_path = './submitB.csv'
submission.to_csv(output_path, index=False)

print(f"Submission saved to: {output_path}")
print(f"Submission shape: {submission.shape}")
print(f"\nFirst 10 predictions:")
print(submission.head(10))

# Also save with probabilities
submission_with_proba = pd.DataFrame({
    'id': test_ids,
    'label': predictions,
    'probability': predictions_proba
})

output_path_proba = './submitB_with_probabilities.csv'
submission_with_proba.to_csv(output_path_proba, index=False)
print(f"\nPredictions with probabilities saved to: {output_path_proba}")

print("\n" + "=" * 80)
print("PIPELINE COMPLETE!")
print("=" * 80)
print(f"Best Model: {best_model_name}")
# print(f"Validation ROC-AUC: {best_auc:.4f}")
print(f"Total Features Used: {len(feature_cols)}")


LOADING TEST DATA
Test data shape: (19999, 87)

Creating features for test data...
Handling missing values in test data...
Test feature matrix shape: (19999, 208)

MAKING PREDICTIONS
Predictions distribution:
Class 0: 13588 (67.9%)
Class 1: 6411 (32.1%)

SAVING RESULTS
Submission saved to: ./submitB.csv
Submission shape: (19999, 2)

First 10 predictions:
                                     id  label
0  256b8b06-2fdb-4d63-8571-8db9babddce9      0
1  6de06e34-05cf-4e41-a583-9f26fb0b01a3      0
2  83f213ef-cf80-4a69-956f-c3c0db5316ae      0
3  131f35e1-32ae-48e4-aca9-f32503d21dc7      1
4  ddcc5e03-e2bb-4b87-8132-5383c63f7386      0
5  31d70b86-b3ad-4e28-a6fa-013efdc40cbb      0
6  b60342e1-f1d9-492d-a7cc-f117b1e66cbe      1
7  c4a0e270-cfb1-4aab-b430-46dc555345b3      0
8  96cd2a19-2a73-495c-9b9e-cf5c7a29d2d1      0
9  c0d9e9a3-0daf-4122-9969-e1d467ca5211      0

Predictions with probabilities saved to: ./submitB_with_probabilities.csv

PIPELINE COMPLETE!
Best Model: stackingClassifier